# Real / Imaginary MP-PCA Denoising

Gaussian noise가 추가된 multi-echo GRE complex 데이터를 Real과 Imaginary 성분으로 나누어 각각 독립적으로 MP-PCA denoising을 수행한다.

Real과 Imaginary 데이터에는 음수 값이 존재할 수 있으므로, DIPY의 `localpca.py`를 기반으로 non-negative clipping 부분을 제거한 `localpca_dn.py`의 `mppca()`를 사용한다.

Denoising 후 Real과 Imaginary를 다시 결합하여 complex 데이터를 생성하고, magnitude 영상을 통해 원본 및 noisy 데이터와 비교한다.


In [ ]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

from localpca_dn import mppca

### 환경설정

Real/Imaginary MP-PCA 실험에 사용할 기본 조건을 설정한다.

- Noise level: 10%, 30%, 50%
- Patch radius: 2
- Patch size: 5 × 5 × 5
- 입력 데이터 경로: `./data/noisy_data`
- 결과 저장 경로: `./data/denoised/real_imag`

모든 noise level에 동일한 patch 크기를 적용하여 조건을 동일하게 유지한다.


### 1. Noise 데이터 불러오기

`Add_noise` 단계에서 생성한 10%, 30%, 50% noisy `.mat` 파일을 불러온다.

각 파일에서 다음 데이터를 사용한다.

- `noisy_real`: noise가 추가된 Real 성분
- `noisy_imag`: noise가 추가된 Imaginary 성분
- `mask_brain`: MP-PCA를 적용할 brain mask
- MRI 메타데이터

불러온 데이터는 noise level별로 구분하여 이후 denoising 과정에서 사용한다.


### 2. Real MP-PCA Denoising

각 noise level의 Real 성분에 수정된 `mppca()`를 적용한다.

Real 데이터에는 음수 값이 정상적으로 존재할 수 있으므로 non-negative clipping을 제거한 `localpca_dn.py`를 사용한다.

10%, 30%, 50% 모두 동일한 5 × 5 × 5 patch를 적용한다.


### 3. Imaginary MP-PCA Denoising

각 noise level의 Imaginary 성분에도 Real과 동일한 조건으로 MP-PCA를 적용한다.

Real과 Imaginary는 서로 합치지 않은 상태에서 각각 독립적으로 denoising한다.


### 4. Real / Imaginary 재결합

각각 MP-PCA를 적용한 Real과 Imaginary 데이터를 다시 결합하여 denoised complex 데이터를 생성한다.

이 단계에서는 Real과 Imaginary의 denoising 결과를 하나의 complex MRI 데이터로 복원한다.


### 5. 결과 저장

10%, 30%, 50%의 Real/Imaginary MP-PCA 결과를 각각 별도의 `.mat` 파일로 저장한다.

저장 파일에는 다음 데이터를 포함한다.

- `deno_real`
- `deno_imag`
- `deno_complex`
- `mask_brain`
- noise level
- patch radius
- MRI 메타데이터

원본 및 noisy 데이터는 기존 파일에 별도로 존재하므로 중복 저장하지 않는다.


### 6. 결과 시각화

Real/Imaginary MP-PCA 결과가 정상적으로 생성되었는지 확인하기 위해 원본, noisy, denoised 영상을 비교한다.

본 실험에서는 예시로 Echo 1, Slice 88을 사용한다.

- Original: noise 추가 전 원본 magnitude
- Noisy: Gaussian noise가 추가된 magnitude
- Denoised Real/Imag: Real과 Imaginary를 각각 MP-PCA 처리한 뒤 다시 결합한 magnitude
- Difference: Denoised와 Original의 차이

Difference 영상을 통해 noise 제거 과정에서 원본 신호가 과도하게 변하지 않았는지 확인한다.


### 최종 확인사항

- 10%, 30%, 50% noisy Real/Imaginary 데이터가 정상적으로 불러와졌는지 확인
- Real과 Imaginary에 각각 독립적으로 MP-PCA가 적용되었는지 확인
- 모든 noise level에 동일한 5 × 5 × 5 patch가 적용되었는지 확인
- Denoised Real과 Imaginary가 정상적으로 complex 데이터로 재결합되었는지 확인
- 각 noise level의 결과 파일이 정상적으로 저장되었는지 확인
- Original / Noisy / Denoised 영상이 정상적으로 표시되는지 확인
- Difference 영상에서 과도한 신호 손실이 나타나지 않는지 확인
